In [ ]:
import os
import re
import PyPDF2
from typing import List, Dict, Tuple, Optional
import json
import time
import multiprocessing
import argparse
import tqdm
import traceback
from concurrent.futures import ProcessPoolExecutor

def extract_text_from_pdf(pdf_path: str) -> List[str]:
    """Extract text from PDF, returning a list of pages."""
    pages = []
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            pages.append(page.extract_text())
    return pages

def extract_case_info(text: str) -> Dict[str, str]:
    """Extract case metadata including claimant name and docket/case number."""
    case_info = {
        "claimant_name": "Unknown",
        "docket_number": "Unknown"
    }
    
    # Extract claimant name - typically appears at the top of the document
    # Pattern 1: Look for typical format in header section
    claimant_pattern1 = r'(.*?)\s*\)\s*\n\s*Claimant'
    claimant_match1 = re.search(claimant_pattern1, text, re.DOTALL)
    
    # Pattern 2: Look for name in title section or near the word "DOCKET"
    claimant_pattern2 = r'([A-Z][A-Z\s\.]+)\s+\d+\s+DOCKET\s+NO\.'
    claimant_match2 = re.search(claimant_pattern2, text)
    
    # Try different patterns for extracting claimant name
    if claimant_match1:
        name = claimant_match1.group(1).strip()
        if not name.isupper() and len(name.split()) <= 1:
            # Likely not a full name
            pass
        else:
            case_info['claimant_name'] = name
    elif claimant_match2:
        case_info['claimant_name'] = claimant_match2.group(1).strip()
        
    # Extract docket/case number
    docket_patterns = [
        r'Docket\s+No\.\s+([0-9,\.]+)',
        r'DOCKET\s+NO\.\s+([0-9,\.]+)',
        r'AP-\d+-\d+-\d+',
        r'CS-\d+-\d+-\d+'
    ]
    
    for pattern in docket_patterns:
        match = re.search(pattern, text)
        if match:
            case_info['docket_number'] = match.group(0)
            break
    
    return case_info

def identify_sections(pages: List[str]) -> List[Tuple[str, str, int, int]]:
    """Identify document sections and their content."""
    sections = []
    combined_text = "\n".join(pages)
    
    # Define standard section headers in workers comp documents
    section_patterns = [
        (r'APPEARANCES', "APPEARANCES"),
        (r'RECORD AND STIPULATIONS', "RECORD AND STIPULATIONS"),
        (r'ISSUES', "ISSUES"),
        (r'FINDINGS OF FACT', "FINDINGS OF FACT"),
        (r'PRINCIPLES OF LAW', "PRINCIPLES OF LAW"),
        (r'PRINCIPLES OF LAW AND ANALYSIS', "PRINCIPLES OF LAW AND ANALYSIS"),
        (r'ANALYSIS', "ANALYSIS"),
        (r'CONCLUSION', "CONCLUSION"),
        (r'CONCLUSIONS', "CONCLUSIONS"),
        (r'AWARD', "AWARD"),
        (r'ORDER', "ORDER"),
        (r'DECISION', "DECISION")
    ]
    
    # Find all section positions
    section_positions = []
    for pattern, section_name in section_patterns:
        for match in re.finditer(r'\b' + pattern + r'\b', combined_text):
            section_positions.append((match.start(), section_name))
    
    # Sort by position
    section_positions.sort(key=lambda x: x[0])
    
    # Extract section content
    for i, (pos, section_name) in enumerate(section_positions):
        start_pos = pos
        if i < len(section_positions) - 1:
            end_pos = section_positions[i+1][0]
        else:
            end_pos = len(combined_text)
        
        # Get the section text
        section_text = combined_text[start_pos:end_pos].strip()
        
        # Get the page number
        page_num = 0
        current_pos = 0
        for i, page in enumerate(pages):
            if start_pos >= current_pos and start_pos < current_pos + len(page):
                page_num = i
                break
            current_pos += len(page) + 1  # +1 for the newline
        
        sections.append((section_name, section_text, page_num, start_pos))
    
    return sections

def create_logical_chunks(text: str, max_tokens: int = 500) -> List[str]:
    """Split text into logical chunks based on paragraphs and size."""
    chunks = []
    paragraphs = re.split(r'\n\s*\n', text)
    
    current_chunk = []
    current_size = 0
    
    for para in paragraphs:
        para_size = len(para.split())
        
        if current_size + para_size > max_tokens and current_chunk:
            # Complete current chunk
            chunks.append('\n\n'.join(current_chunk))
            current_chunk = [para]
            current_size = para_size
        else:
            current_chunk.append(para)
            current_size += para_size
    
    # Add the final chunk if it exists
    if current_chunk:
        chunks.append('\n\n'.join(current_chunk))
    
    return chunks

def process_workers_comp_document(pdf_path: str, max_tokens: int = 500) -> List[Dict]:
    """Process a workers comp PDF and create chunks with metadata."""
    pages = extract_text_from_pdf(pdf_path)
    
    # Extract document metadata
    metadata = extract_case_info("\n".join(pages[:2]))  # Use first two pages to find metadata
    
    # Identify document sections
    sections = identify_sections(pages)
    
    # Create chunks for each section with appropriate metadata
    all_chunks = []
    
    for section_name, section_text, page_num, _ in sections:
        # Skip section headers that are too short or just contain the header
        if len(section_text.strip()) <= len(section_name) + 5:
            continue
            
        # Create logical chunks for this section
        chunks = create_logical_chunks(section_text, max_tokens)
        
        for i, chunk_text in enumerate(chunks):
            chunk_data = {
                "text": f"Case: {metadata['docket_number']} | Claimant: {metadata['claimant_name']}\nSection: {section_name}\n\n{chunk_text}",
                "metadata": {
                    "docket_number": metadata['docket_number'],
                    "claimant_name": metadata['claimant_name'],
                    "section": section_name,
                    "page": page_num,
                    "chunk": i,
                    "filename": os.path.basename(pdf_path)
                }
            }
            all_chunks.append(chunk_data)
    
    return all_chunks

def process_directory(directory: str, output_file: str, max_tokens: int = 500):
    """Process all PDFs in a directory and save chunks to a JSONL file."""
    all_chunks = []
    
    for filename in os.listdir(directory):
        if filename.lower().endswith('.pdf'):
            pdf_path = os.path.join(directory, filename)
            try:
                chunks = process_workers_comp_document(pdf_path, max_tokens)
                all_chunks.extend(chunks)
                print(f"Processed {filename}: {len(chunks)} chunks created")
            except Exception as e:
                print(f"Error processing {filename}: {str(e)}")
    
    # Write chunks to JSONL file
    with open(output_file, 'w') as f:
        for chunk in all_chunks:
            f.write(json.dumps(chunk) + '\n')
    
    print(f"Total chunks written: {len(all_chunks)}")

def postprocess_cleanup(jsonl_path: str, output_path: str):
    """Additional cleaning of the JSONL file to improve quality."""
    cleaned_chunks = []
    
    with open(jsonl_path, 'r') as f:
        for line in f:
            chunk = json.loads(line)
            
            # Clean text content
            text = chunk['text']
            
            # Remove excessive whitespace
            text = re.sub(r'\s+', ' ', text)
            
            # Fix common OCR errors
            text = re.sub(r'(\w)- (\w)', r'\1\2', text)  # Remove hyphenation
            
            # Update the chunk
            chunk['text'] = text
            cleaned_chunks.append(chunk)
    
    # Write cleaned chunks
    with open(output_path, 'w') as f:
        for chunk in cleaned_chunks:
            f.write(json.dumps(chunk) + '\n')

# Enhanced functions from ks-legal-chunkier.py
def extract_enhanced_case_info(pages: List[str]) -> Dict[str, str]:
    """Better extraction of worker comp case information."""
    full_text = "\n".join(pages[:3])  # Use first few pages
    
    info = {
        "claimant_name": "Unknown",
        "docket_number": "Unknown",
        "respondent_name": "Unknown"
    }
    
    # Extract claimant
    claimant_patterns = [
        r'([A-Z][A-Z\s\.]+)\s+\)\s*\n\s*Claimant',
        r'([A-Z][A-Z\s\.]+)\s+\d+\s+DOCKET\s+NO\.',
        r'(CARI\s+PRA\s*TER)\s+\d+\s+AP-\d+-\d+-\d+', # Match the specific format in this document
        r'ANTHONY L\. MARCEAU',  # Specific pattern in your example
        r'LOREN ALBRIGHT',       # Specific pattern in your example
        r'CARI PRATER'           # Specific pattern in your example
    ]
    
    for pattern in claimant_patterns:
        match = re.search(pattern, full_text)
        if match:
            info['claimant_name'] = match.group(1).strip()
            break
    
    # Extract docket number with various formats
    docket_patterns = [
        r'Docket\s+No\.\s+([\d,\.]+)',
        r'DOCKET\s+NO\.\s+([\d,\.]+)',
        r'(AP-\d+-\d+-\d+)',
        r'(CS-\d+-\d+-\d+)',
        r'(231,675)',  # Specific number in example
        r'(152,410)'   # Specific number in example
    ]
    
    for pattern in docket_patterns:
        match = re.search(pattern, full_text)
        if match:
            info['docket_number'] = match.group(1).strip()
            break
    
    # Extract respondent
    respondent_patterns = [
        r'VS\.\s*\)\s*\n\s*\)\s*\n\s*([A-Z][A-Z\s\.]+)',
        r'VS\.\s*\)\s*\n\s*([A-Z][A-Z\s\.]+)'
    ]
    
    for pattern in respondent_patterns:
        match = re.search(pattern, full_text)
        if match:
            info['respondent_name'] = match.group(1).strip()
            break
    
    return info

def process_pdf_file(pdf_path, file_index=None, total_files=None):
    """Process a single PDF file and return its chunks."""
    try:
        filename = os.path.basename(pdf_path)
        pages = extract_text_from_pdf(pdf_path)
        
        # Get enhanced metadata
        metadata = extract_enhanced_case_info(pages)
        
        # Identify sections using better pattern matching
        sections = []
        combined_text = "\n".join(pages)
        
        section_patterns = [
            r'\bAPPEARANCES\b',
            r'\bRECORD AND STIPULATIONS\b',
            r'\bISSUES\b',
            r'\bFINDINGS OF FACT\b',
            r'\bPRINCIPLES OF LAW( AND ANALYSIS)?\b',
            r'\bANALYSIS\b',
            r'\bCONCLUSION(S)?\b',
            r'\bAWARD\b',
            r'\bORDER\b',
            r'\bDECISION\b'
        ]
        
        # Find all section positions
        section_positions = []
        for pattern in section_patterns:
            for match in re.finditer(pattern, combined_text):
                section_name = match.group(0)
                section_positions.append((match.start(), section_name))
        
        # Sort by position
        section_positions.sort(key=lambda x: x[0])
        
        file_chunks = []
        # Extract sections
        for i, (pos, section_name) in enumerate(section_positions):
            start_pos = pos
            if i < len(section_positions) - 1:
                end_pos = section_positions[i+1][0]
            else:
                end_pos = len(combined_text)
            
            section_text = combined_text[start_pos:end_pos].strip()
            
            # Create chunks of appropriate size
            paragraphs = re.split(r'\n\s*\n', section_text)
            current_chunk = []
            current_size = 0
            max_size = 800  # Tokens/words
            
            for para in paragraphs:
                para_size = len(para.split())
                
                if current_size + para_size > max_size and current_chunk:
                    # Finalize this chunk
                    chunk_text = "\n\n".join(current_chunk)
                    chunk_data = {
                        "text": f"Case: {metadata['docket_number']} | Claimant: {metadata['claimant_name']}\nSection: {section_name}\n\n{chunk_text}",
                        "metadata": {
                            "docket_number": metadata['docket_number'],
                            "claimant_name": metadata['claimant_name'],
                            "section": section_name,
                            "filename": filename
                        }
                    }
                    file_chunks.append(chunk_data)
                    
                    # Start new chunk
                    current_chunk = [para]
                    current_size = para_size
                else:
                    current_chunk.append(para)
                    current_size += para_size
            
            # Add final chunk if it exists
            if current_chunk:
                chunk_text = "\n\n".join(current_chunk)
                chunk_data = {
                    "text": f"Case: {metadata['docket_number']} | Claimant: {metadata['claimant_name']}\nSection: {section_name}\n\n{chunk_text}",
                    "metadata": {
                        "docket_number": metadata['docket_number'],
                        "claimant_name": metadata['claimant_name'],
                        "section": section_name,
                        "filename": filename
                    }
                }
                file_chunks.append(chunk_data)
        
        return {
            "status": "success",
            "filename": filename,
            "chunks": file_chunks,
            "metadata": metadata
        }
    
    except Exception as e:
        error_details = traceback.format_exc()
        return {
            "status": "error",
            "filename": os.path.basename(pdf_path),
            "error": str(e),
            "details": error_details
        }

def enhanced_workers_comp_processor(directory: str, output_file: str, workers: int = 4, resume: bool = False, pattern: Optional[str] = None, verbose: bool = False):
    """Enhanced processor with parallel processing and resumable capability."""
    start_time = time.time()
    processed_files = set()
    all_chunks = []
    errors = []
    
    # Check if we need to resume from a previous run
    if resume and os.path.exists(output_file):
        try:
            with open(output_file, 'r') as f:
                for line in f:
                    chunk = json.loads(line)
                    all_chunks.append(chunk)
                    processed_files.add(chunk["metadata"]["filename"])
            print(f"Resuming from previous run. {len(processed_files)} files already processed.")
        except Exception as e:
            print(f"Error loading previous results: {e}")
            print("Starting fresh processing run.")
    
    # Get list of PDF files to process
    pdf_files = []
    for filename in os.listdir(directory):
        if filename.lower().endswith('.pdf') and filename not in processed_files:
            if pattern is None or re.search(pattern, filename):
                pdf_files.append(os.path.join(directory, filename))
    
    if not pdf_files:
        print("No new files to process.")
        return
    
    print(f"Processing {len(pdf_files)} PDF files with {workers} parallel workers...")
    
    # Create a checkpoint file to store progress
    checkpoint_file = f"{output_file}.checkpoint"
    log_file = f"{output_file}.log"
    
    # Process files in parallel
    with ProcessPoolExecutor(max_workers=workers) as executor:
        results = list(tqdm.tqdm(
            executor.map(process_pdf_file, pdf_files), 
            total=len(pdf_files),
            desc="Processing PDFs"
        ))
    
    # Process results
    for result in results:
        if result["status"] == "success":
            all_chunks.extend(result["chunks"])
            if verbose:
                print(f"Processed {result['filename']} - Claimant: {result['metadata']['claimant_name']}, "
                      f"Docket: {result['metadata']['docket_number']}, Chunks: {len(result['chunks'])}")
        else:
            errors.append(result)
            print(f"Error processing {result['filename']}: {result['error']}")
            # Write detailed error to log
            with open(log_file, "a") as log:
                log.write(f"\n--- Error processing {result['filename']} ---\n")
                log.write(result["details"])
                log.write("\n-----------------------\n")
    
    # Write all chunks to JSONL
    with open(output_file, 'w') as f:
        for chunk in all_chunks:
            f.write(json.dumps(chunk) + '\n')
    
    end_time = time.time()
    duration = end_time - start_time
    
    print(f"Processing complete!")
    print(f"Total chunks written: {len(all_chunks)}")
    print(f"Total files processed: {len(pdf_files)}")
    print(f"Files with errors: {len(errors)}")
    print(f"Duration: {duration:.2f} seconds ({duration/60:.2f} minutes)")

def list_pdf_files(directory: str, pattern: Optional[str] = None):
    """List PDF files in directory, optionally filtering by pattern."""
    files = []
    for filename in os.listdir(directory):
        if filename.lower().endswith('.pdf'):
            if pattern is None or re.search(pattern, filename):
                files.append(filename)
    
    print(f"Found {len(files)} PDF files {f'matching pattern {pattern}' if pattern else ''}:")
    for filename in sorted(files):
        print(f"  - {filename}")
    
    return files

def process_single_file(file_path: str, output_file: str):
    """Process a single PDF file and save chunks to JSONL."""
    if not os.path.exists(file_path):
        print(f"Error: File {file_path} does not exist.")
        return
        
    print(f"Processing single file: {file_path}")
    
    result = process_pdf_file(file_path)
    
    if result["status"] == "success":
        with open(output_file, 'w') as f:
            for chunk in result["chunks"]:
                f.write(json.dumps(chunk) + '\n')
        print(f"Successfully processed {result['filename']} - Generated {len(result['chunks'])} chunks")
    else:
        print(f"Error processing {result['filename']}: {result['error']}")

# Example usage
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description='Process workers compensation PDF documents in parallel.')
    parser.add_argument('--directory', '-d', type=str, 
                        default=r'H:\My Drive\Kansas Workers Compensation Data\KS WC Appeal Decisions',
                        help='Directory containing PDF files to process')
    parser.add_argument('--output', '-o', type=str, 
                        default="workers_comp_chunks.jsonl",
                        help='Output JSONL file')
    parser.add_argument('--workers', '-w', type=int, default=4,
                        help='Number of parallel workers')
    parser.add_argument('--resume', '-r', action='store_true',
                        help='Resume from previous run')
    parser.add_argument('--list', '-l', action='store_true',
                        help='List PDF files in directory instead of processing')
    parser.add_argument('--pattern', '-p', type=str, default=None,
                        help='Regex pattern to filter PDF files (e.g., "2023" for files with 2023 in the name)')
    parser.add_argument('--file', '-f', type=str, default=None,
                        help='Process a single specific PDF file')
    parser.add_argument('--verbose', '-v', action='store_true',
                        help='Enable verbose output')
    
    args = parser.parse_args()
    
    # Just list files if requested
    if args.list:
        list_pdf_files(args.directory, args.pattern)
    # Process a single file if specified
    elif args.file:
        file_path = args.file if os.path.exists(args.file) else os.path.join(args.directory, args.file)
        process_single_file(file_path, args.output)
    # Process directory
    else:
        enhanced_workers_comp_processor(
            args.directory, 
            args.output,
            workers=args.workers,
            resume=args.resume,
            pattern=args.pattern,
            verbose=args.verbose
        )